# Week 7: How Canadian is the Giller Prize?

The Scotiabank Giller Prize is awarded each year to the best Canadian novel or short story collection. But what makes a writer "Canadian"? Being born here? Living here? And within Canada, does the prize reach writers from across the country, or does it keep finding them in the same few places?

This week we use the `Born` and `Residence` columns of our Giller data set to ask two questions:

1. **How Canadian are the nominees?** Where were they born, and where do they live now?
2. **Where in Canada do the nominees come from?** Which provinces were they born in, and how does that compare with the number of people who live in each province?

Remember that our data set contains every *nominated* book (longlist and shortlist), not only the winners.

Review from Week 6:
- Reading each row as a dictionary with `csv.DictReader`
- Splitting and stripping strings with `.split()` and `.strip()`
- Counting with a dictionary
- Recording cleaning decisions in a dictionary of corrections
- Writing functions
- Checking that a value is not in a list with `not in` (Week 4)

New Python concepts:
- Splitting a string on a word, not just a single character
- Combining two conditions with `and` and `or`
- Building a new dictionary of cleaned values from an existing dictionary

## 1. Loading the clean data

We'll start from the file `giller_nominees_clean.csv` that we saved last week, with the stray spaces already removed. It is in the same folder as this notebook.

In [ ]:
import csv

with open("giller_nominees_clean.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    nominees = list(reader)

print(len(nominees), "nominated books")

## 2. Where were the nominees born?

Now let's try to answer a question about geography. The `Born` column holds values like `"Colombo, Sri Lanka"` and `"Petrolia, ON"`: a city followed by a province, state, or country. To ask questions like "how many nominees were born outside Canada?" we need to **split the city from the region**.

### Step 1: One birthplace per author

The rows in our data set are *books*, not *authors*. If we counted birthplaces row by row, an author with 6 nominations would be counted 6 times. So first we build a dictionary that maps each author to their birthplace. Remember that dictionary keys are unique, so each author appears only once, however many books they have.

Note that when an author appears more than once in the data set, our solution below will overwrite or replace the birthplace value for that author.  This should be okay because an author doesn't have different birthplaces (unless there is an error in the data set.)

In [ ]:
# Create an empty dictionary 
birthplaces = {}

# For each row in the list of nominees
for book in nominees:
    # Get the birth place and name of the author of this book
    born = book["Born"]  
    author = book["Author"]

    # Add the birthplace of the author to the dictionary
    birthplaces[author] = born

# How many unique authors are in the data set?
print(len(birthplaces), "authors")
print(birthplaces["Shyam Selvadurai"])

### Step 2: Splitting city from region

If we want to group together authors by their region, then we need to separate the city from the region (province or country). Looking at this column we see values like: "Brampton, ON", "London, England", "Baltimore, MD, USA", "South Africa", "Ontario".  We can see the region comes after the last comma, unless there is no comma, in which case the region is the whole string.

`.split(",")` works well here: the `csv` module has already dealt with the quotes, so any comma left inside a value really does separate the city from the region.

In [ ]:
born = birthplaces["Shyam Selvadurai"]
parts = born.split(",")
print(parts)

Notice the space at the start of `' Sri Lanka'`: we'll need `.strip()` again. Also, some birthplaces have more than one comma, and some have none at all:

In [ ]:
print(birthplaces["Joseph Boyden"].split(","))
print(birthplaces["Timothy Taylor"].split(","))

Using index `-1` always gives us the **last** piece, which is the region we want, no matter how many commas there are. And if there's no comma, the whole value is the region.

Let's put these steps together with an example value for `place`.

In [ ]:
# The region is the last item in the list, and we want to remove
# the whitespace around it.
region = parts[-1].strip()
print(region)

Now that we know how to obtain the region from a birth place, let's make it a function so that we can reuse this code.

In [ ]:
def get_region(place):
    """ Return the province, state or country at the end of place.

        Example: get_region("Colombo, Sri Lanka") returns "Sri Lanka"
    """
    parts = place.split(",")   
    region = parts[-1].strip()
    
    return region


print(get_region("Colombo, Sri Lanka"))
print(get_region("Willowdale, North York, ON"))
print(get_region("Venezuela"))

### Step 3: Count the regions, and look at what we got

Let's write write the counting loop over the `birthplaces` dictionary to count the number of regions. To do this we will create another dictionay to hold the count of each region that we find, so that we know how many authors were born in each region.

Recall that `birthplaces` authors to their birthplaces.  In other words the keys in the dictionary `birthplaces` are the authors and the values in the dictionary are the places the authors were born.

This is a loop pattern we've seen before and a very common one!

In [ ]:
region_counts = {}

for author in birthplaces:

    region = get_region(birthplaces[author])

    if region == '-':
        print(author + " has no region")

    # If we've seen the region before, add 1 to its count
    if region in region_counts:
        region_counts[region] = region_counts[region] + 1
    else:
        region_counts[region] = 1

for region in region_counts:
    print(region, region_counts[region])

Read through this list carefully. Before we start aggregating this data, we need to decide how to deal with several problems. For example:

- **The same place written different ways:** `Ontario` and `ON`, `Alberta` and `AB`, `British Columbia` and `BC`. 
- **Mixed levels:** Canadian provinces appear next to whole countries (`England`, `Kenya`, `USA`). For authors born in Canada the spreadsheet records the province, but for everyone else it records the country. (Because `get_region()` keeps the last piece, `"Oak Park, IL, USA"` gives `USA`, not the state.) Python has no way of knowing that `ON` is part of Canada while `Kenya` is a country of its own; we'll have to tell it.
- **Missing detail:** one birthplace is just `Canada`. We know the author was born in Canada, but not in which province.
- **Historical names:** `West Germany`, `South Vietnam`, and `Yugoslavia` no longer exist, and the political status of `Tibet` is contested. Should an author born in West Germany in 1960 count as born in "Germany"? There isn't one right answer, but **we** have to decide, and we should say what we decided.
- **Placeholders:** a `-` means "unknown." (We'll skip these when we count.)

### Step 4: Fixing values with a dictionary

A dictionary is a handy way to record our cleaning decisions: the key is the messy value, and the value is what we want to replace it with. Writing the decisions down in one place also makes them easy for someone else to check and argue with.

We can use this to address the problem of provinces being written in two ways.

In [ ]:
corrections = {
    # Full province names: use the abbreviations found everywhere else
    "Ontario": "ON",
    "Alberta": "AB",
    "British Columbia": "BC",
}

# Trying out a correction
region = "Ontario"
if region in corrections:
    region = corrections[region]
print(region)

If `region` is a key in `corrections`, we replace it with the corrected value. If it isn't (try changing the value of `region` from `"Ontario"` to `"Kenya"`), the `if` does nothing and `region` keeps its value.

Notice what we did **not** put in `corrections`: we have left historical names like `West Germany` exactly as they were recorded. That is a judgment call too, and it doesn't affect the question we are about to ask.

Some problems shouldn't be "fixed" in our code at all. A `-` means the birthplace is unknown, and the right fix is to find the information and enter it in the spreadsheet. In our code, we'll **skip** values we can't use and print them so that we can report them.

### Step 5: Clean each birthplace once

We now have everything we need to clean a birthplace: `get_region()` finds the region, and `corrections` fixes it. Rather than repeating these steps every time we ask a new question, we'll do them **once** for every author and store the results in a new dictionary, `born_regions`. Its keys are the authors, just like `birthplaces`, but its values are the cleaned regions.

We keep the `-` values in `born_regions` for now, so that every author has an entry. We'll skip them when we count.

In [ ]:
born_regions = {}

for author in birthplaces:
    region = get_region(birthplaces[author])
    if region in corrections:
        region = corrections[region]
    born_regions[author] = region

print(birthplaces["Shyam Selvadurai"], "->", born_regions["Shyam Selvadurai"])
print(birthplaces["Loghan Paylor"], "->", born_regions["Loghan Paylor"])

From now on, whenever we need an author's birth region we can look it up in `born_regions`. Counting the regions is the same counting loop we have used before:

In [ ]:
born_region_counts = {}

for author in born_regions:
    region = born_regions[author]

    if region == "-":
        print(f"Skipping {author}: birthplace unknown")
    elif region in born_region_counts:
        born_region_counts[region] = born_region_counts[region] + 1
    else:
        born_region_counts[region] = 1

print()
for region in born_region_counts:
    print(region, born_region_counts[region])

### Step 6: Born in Canada, or elsewhere?

Finally, let's group the regions into two categories. A list of the Canadian province and territory abbreviations, together with the `in` operator, does the job. We also include `"Canada"` itself, for the author whose birthplace doesn't name a province and "Treaty 6 Territory".  Treaty 6 Territory spans Alberta and Saskatchewan, so we have to make a decision about how to record it. We are making a judgment call to include this author among those born in Canada.

There is a piece of Python syntax in the print statement that we haven't talked about: `:.0f`.  This is called a *format specifier* and it is used in this case to suppress printing a whole bunch of decimal places. Don't worry about memorizing this.  This is the kind of detail to look up when you need it.

In [ ]:
canadian_regions = ["AB", "BC", "MB", "NB", "NL", "NS", "NT", "NU", "ON", "PE", "QC", "SK", "YT", "Canada", "Treaty 6 Territory"]

born_in_canada = 0
born_elsewhere = 0

for author in born_regions:
    region = born_regions[author]
    if region in canadian_regions:
        born_in_canada = born_in_canada + 1
    elif region != "-":   # skip unknown birthplaces
        born_elsewhere = born_elsewhere + 1

total = born_in_canada + born_elsewhere
print(f"Born in Canada:    {born_in_canada} ({born_in_canada / total * 100:.0f}%)")
print(f"Born elsewhere:    {born_elsewhere} ({born_elsewhere / total * 100:.0f}%)")

**Discussion:**
- About a quarter of the nominees we have data for were born outside Canada. How might you include this information in a discussion of what "counts" as a Canadian Literature?
- Three authors' birthplaces are unknown (`-`). Could they change the result? More generally, which authors are most likely to have missing biographical information, and why does that matter?
- Every cleaning decision we made (counting `Treaty 6 Territory` as Canada, leaving `West Germany` as recorded, skipping unknown values, counting authors rather than books) changes the final numbers a little. Where should we record those decisions so that a reader of our analysis can decide for themselves how reasonable our decisions were?

## 3. Where do the nominees live now?

Being born in Canada is only one way of being Canadian. Many authors move here as children or as adults, and some authors born in Canada now live somewhere else. The `Residence` column records where each author lives.

We'll follow the same steps we used for birthplaces, starting with one residence per author.

In [ ]:
residences = {}

for book in nominees:
    residence = book["Residence"]
    if residence != "":
        residences[book["Author"]] = residence

print(len(residences), "authors with a residence")
print(residences["Shani Mootoo"])

### Step 1: Look before you count

Before we count anything, it's worth looking for values that won't fit our method. Some authors live in more than one place. We can find them by checking whether the word `" and "` appears in the value. (Remember that `in` works on strings too: it checks whether one string appears inside another.)

In [ ]:
for author in residences:
    if " and " in residences[author]:
        print(f"{author}: {residences[author]}")

If we used `get_region()` on `"Montreal, QC and Paris, France"`, we would get `France`, and lose Montreal completely. We need a rule, and once again it's a decision a person has to make, not Python.

**Our rule: use the first place listed.** It's simple, and it's easy to explain to a reader. (Is it the best rule? We'll come back to that in the discussion.)

So far we have only split strings on a comma, but `.split()` can split on any string we give it, including a whole word with spaces around it. Splitting on `" and "` and keeping item `0` gives us the first place.

In [ ]:
place = "Montreal, QC and Paris, France"
parts = place.split(" and ")
print(parts)
print(parts[0])

place = "Toronto, ON"
parts = place.split(" and ")
print(parts[0])   # only one place, so nothing changes

We will only use this rule for residences. `"Trinidad and Tobago"` appears in the birthplace list and a single country whose name happens to contain `" and "`, so we shouldn't use this method for birthplaces!

### Step 2: Look at the regions

Just as we did for birthplaces, let's count the regions before cleaning them, so that we can see what problems they contain. For each author we take the first place listed, then use `get_region()`.

In [ ]:
region_counts = {}

for author in residences:
    parts = residences[author].split(" and ")
    region = get_region(parts[0])
    if region in region_counts:
        region_counts[region] = region_counts[region] + 1
    else:
        region_counts[region] = 1

for region in region_counts:
    print(region, region_counts[region])

What problems do you see this time? Some are familiar from the birthplaces, and some are new:

- **Full names instead of abbreviations:** `New Brunswick`, `Newfoundland`, `Nova Scotia`, `Quebec`, and `South-Eastern Ontario`.
- **The United States written two ways:** `USA` and `United States`.
- **Leftovers from our "first place" rule:** `Galiano Island` comes from `"Galiano Island and Victoria, BC"`. The first place has no province of its own! 
- **Placeholders:** one residence is `-`, which we'll skip.

### Step 3: Adding to our corrections

We don't need a new dictionary for these. We can add new keys to the `corrections` dictionary we made for birthplaces. Assigning to a key that isn't in a dictionary yet adds it.

In [ ]:
# Full names: use the abbreviations found everywhere else
corrections["New Brunswick"] = "NB"
corrections["Newfoundland"] = "NL"
corrections["Nova Scotia"] = "NS"
corrections["Quebec"] = "QC"
corrections["South-Eastern Ontario"] = "ON"

# The United States, written out in full
corrections["United States"] = "USA"

# Leftovers from the "first place" rule
corrections["Galiano Island"] = "BC"   # an island in BC

print(len(corrections), "corrections")

### Step 4: Clean each residence once

Now we build `home_regions` in the same way we built `born_regions`, using the first place listed. We added the new corrections *before* running this loop, so they are used.

In [ ]:
home_regions = {}

for author in residences:
    parts = residences[author].split(" and ")
    region = get_region(parts[0])
    if region in corrections:
        region = corrections[region]
    home_regions[author] = region

print(residences["Michael Winter"], "->", home_regions["Michael Winter"])

In [ ]:
home_region_counts = {}

for author in home_regions:
    region = home_regions[author]

    if region == "-":
        print(f"Skipping {author}: residence unknown")
    elif region in home_region_counts:
        home_region_counts[region] = home_region_counts[region] + 1
    else:
        home_region_counts[region] = 1

print()
for region in home_region_counts:
    print(region, home_region_counts[region])

We have left `England`, `Scotland`, and `United Kingdom` as separate regions. Should they be combined? For the question we're about to ask, it doesn't matter, because none of them is in Canada.

### Step 5: Living in Canada, or elsewhere?

This is the same calculation we did for birthplaces, using the same `canadian_regions` list.

In [ ]:
lives_in_canada = 0
lives_elsewhere = 0

for author in home_regions:
    region = home_regions[author]
    if region in canadian_regions:
        lives_in_canada = lives_in_canada + 1
    elif region != "-":   # skip unknown residences
        lives_elsewhere = lives_elsewhere + 1

total = lives_in_canada + lives_elsewhere
print(f"Lives in Canada:   {lives_in_canada} ({lives_in_canada / total * 100:.0f}%)")
print(f"Lives elsewhere:   {lives_elsewhere} ({lives_elsewhere / total * 100:.0f}%)")

## 4. How Canadian? Putting birth and residence together

About three quarters of the nominees were born in Canada, and about five in six live here now. But those two numbers don't tell us whether they are the *same* authors. To find out, we need to look at each author's birthplace and residence together. Every author falls into one of four groups:

|                      | **Lives in Canada**       | **Lives elsewhere**        |
|----------------------|---------------------------|----------------------------|
| **Born in Canada**   | born here, lives here     | born here, lives elsewhere |
| **Born elsewhere**   | born elsewhere, lives here| born elsewhere, lives elsewhere |

To sort authors into these groups we need to check **two conditions at once**. We saw the keyword `and` briefly in Week 4: a condition like `A and B` is `True` only when `A` and `B` are **both** `True`.

Each `elif` in the code below matches one cell of the table. For "elsewhere" we use `not in`, which we saw in Week 4: `home not in canadian_regions` is `True` when `home` is **not** in the list.

In [ ]:
born_here_lives_here = 0
born_here_lives_elsewhere = 0
born_elsewhere_lives_here = 0
born_elsewhere_lives_elsewhere = []   # a list, so that we can look at the names

for author in born_regions:
    # check this author has a residence too, so we don't get a KeyError
    if author in home_regions:
        born = born_regions[author]
        home = home_regions[author]

        if born == "-" or home == "-":
            print(f"Skipping {author}: birthplace or residence unknown")
        elif born in canadian_regions and home in canadian_regions:
            born_here_lives_here = born_here_lives_here + 1
        elif born in canadian_regions and home not in canadian_regions:
            born_here_lives_elsewhere = born_here_lives_elsewhere + 1
        elif born not in canadian_regions and home in canadian_regions:
            born_elsewhere_lives_here = born_elsewhere_lives_here + 1
        else:
            # the only group left: born elsewhere and lives elsewhere
            born_elsewhere_lives_elsewhere.append(author)

neither = len(born_elsewhere_lives_elsewhere)
total = born_here_lives_here + born_here_lives_elsewhere + born_elsewhere_lives_here + neither

print()
print(f"Born in Canada, lives in Canada:   {born_here_lives_here} ({born_here_lives_here / total * 100:.0f}%)")
print(f"Born in Canada, lives elsewhere:   {born_here_lives_elsewhere} ({born_here_lives_elsewhere / total * 100:.0f}%)")
print(f"Born elsewhere, lives in Canada:   {born_elsewhere_lives_here} ({born_elsewhere_lives_here / total * 100:.0f}%)")
print(f"Born elsewhere, lives elsewhere:   {neither} ({neither / total * 100:.0f}%)")

The first line of our `if` uses `or`, the partner of `and`. A condition like `A or B` is `True` when **at least one** of `A` and `B` is `True`. Here, we skip an author if *either* of their places is unknown.

Who are the authors in the last group?

In [ ]:
for author in born_elsewhere_lives_elsewhere:
    print(f"{author}: born {birthplaces[author]}; lives {residences[author]}")

**Discussion:**
- Only about two thirds of the nominees were born in Canada *and* live here now. Is that more or less than you expected for a prize for Canadian fiction?
- The Giller is open to Canadian citizens and permanent residents. Look up two or three of the authors in the last group. What makes them eligible? What does their presence on the list suggest about what "Canadian literature" means?
- About one in eight nominees was born in Canada but lives elsewhere, and about one in five was born elsewhere but lives here. Do these two groups tell different stories about Canadian writing?
- Our "first place listed" rule put Cheluchi Onyemeluke-Onubi (`"Lagos, Nigeria and Halifax, Nova Scotia."`) and André Forget (`"Sheffield, England and Toronto, Canada."`) outside Canada. What would be a fairer rule? How would you write it in Python?
- `Residence` records where each author lives *now*, not where they lived when they were nominated. How might that affect our results, especially for nominees from the 1990s?

## 5. Which provinces were the nominees born in?

In section 2 we found that about three quarters of the nominees were born in Canada. Now let's look more closely: **which provinces** were they born in?

We can already guess that Ontario will come first. But that doesn't tell us much on its own, because Ontario is also where almost 4 in 10 Canadians live. A province with more people will produce more writers, and more nominees, even if nothing special is happening there. To find out whether a province produces *more* nominees than we would expect, we need to compare the number of nominees with the number of **people**.

### Step 1: Recording the populations

We'll record each province's and territory's population in a dictionary. The keys are the same abbreviations we use in `born_regions`, which will make it easy to look a population up. These populations are from Statistics Canada's 2021 Census. We have listed the provinces from west to east, followed by the territories.

In [ ]:
province_populations = {
    "BC": 5000879,
    "AB": 4262635,
    "SK": 1132505,
    "MB": 1342153,
    "ON": 14223942,
    "QC": 8501833,
    "NB": 775610,
    "NS": 969383,
    "PE": 154331,
    "NL": 510550,
    "YT": 40232,
    "NT": 41070,
    "NU": 36858,
}

# Add up the populations to get the population of Canada
canada_population = 0
for province in province_populations:
    canada_population = canada_population + province_populations[province]

print(canada_population)

That total matches the population of Canada in the 2021 Census, so we haven't left anyone out.

### Step 2: Counting nominees by province of birth

This is our usual counting pattern, with two small changes:

- We start every province's count at `0`, so that a province with no nominees still shows up in our results. (A `0` is a result too!) Because every province is already a key, we can always add 1, without needing an `else`.
- We only count an author if their region is a key in `born_province_counts`. That skips the authors born outside Canada and those whose birthplace is unknown (`-`). It also skips the one author whose birthplace is recorded only as `Canada`, because we don't know which province to count them in.

In [ ]:
born_province_counts = {}

# Start every province at 0
for province in province_populations:
    born_province_counts[province] = 0

# Count the nominees born in each province
for author in born_regions:
    region = born_regions[author]
    if region in born_province_counts:
        born_province_counts[region] = born_province_counts[region] + 1

# Add up the counts to get the number of nominees we counted
born_in_a_province = 0
for province in born_province_counts:
    born_in_a_province = born_in_a_province + born_province_counts[province]

print(born_in_a_province, "nominees born in a province or territory")
print()
for province in born_province_counts:
    print(province, born_province_counts[province])

Ontario has by far the most nominees. Does that mean Ontario is especially good at producing Giller nominees? Not necessarily: it also has by far the most people.

### Step 3: Comparing shares

A fairer comparison asks: what **share** of the nominees was born in each province, and what share of Canadians lives there? If a province's share of nominees is about the same as its share of the population, it produces about as many nominees as we would expect for its size.

In [ ]:
for province in province_populations:
    nominee_share = born_province_counts[province] / born_in_a_province * 100
    population_share = province_populations[province] / canada_population * 100
    print(f"{province}: {nominee_share:4.1f}% of nominees, {population_share:4.1f}% of Canadians")

(The `4` in `:4.1f` pads each number to at least 4 characters wide, so the columns line up.)

Now we can see what the raw counts hid. Ontario's share of the nominees is almost exactly its share of the population. Newfoundland and Labrador's share of the nominees is more than four times its share of the population, while Quebec's is well below its share.

### Step 4: Nominees per million people

Another common way to compare places of different sizes is a **rate**: how many nominees there are for every million people who live there. Newspapers use the same idea when they report crimes or hospital beds "per 100,000 people".

In [ ]:
for province in province_populations:
    per_million = born_province_counts[province] / province_populations[province] * 1000000
    print(f"{province}: {born_province_counts[province]:2} nominees, {per_million:4.1f} per million people")

A map can show these rates at a glance. On the left, each province is shaded by its number of nominees per million people, using where the nominees were **born**; the darker the blue, the higher the rate. On the right is the same map for where the nominees **live now**. The territories are hatched in gray, because they have too few people for a reliable rate.

Next week we will show how to create a graph like this one, but for now we can enjoy the output.

![Two maps of Canada shading each province by Giller nominees per million people, for where nominees were born and where they live now. Newfoundland and Labrador is darkest on both.](Nominees_by_province.png)

Nunavut now has the highest rate of all, from a single nominee! When the numbers are this small, one author more or less makes a huge difference to a rate. That is one reason to always report the count alongside a rate or a percentage, as we did here.

**Discussion:**
- Newfoundland and Labrador produces far more nominees than its population would suggest. What might explain that? What else would you want to know before making a claim about it?
- Quebec produces fewer nominees than its population would suggest. The Giller is a prize for books published in English, including translations. How might that affect the result for Quebec? Does it mean our comparison with the whole population of Quebec is unfair?
- Prince Edward Island and the territories have only 0 or 1 nominees each. How many nominees do you think a province needs before its percentage or rate tells us something?
- We compared birthplaces with **today's** populations, but most nominees were born decades ago, when the provinces' shares of Canada's population were different. How might that change the comparison?

## 6. Saving our cleaned table

We've made a lot of decisions this week: one row per author, the "first place listed" rule, the corrections dictionary, and which regions count as Canada. Let's save the results as a new CSV file, so that we (and other people) can use the cleaned data without redoing all of this work. Just as in Week 6, the code is a record of exactly what we changed.

Our new table will have **one row per author**, with these columns:

| Column | What it holds |
|---|---|
| `Author` | the author's name |
| `Born` | the birthplace, as recorded in the spreadsheet |
| `Born region` | the cleaned province, state, or country |
| `Born in Canada` | `Canada`, `Elsewhere`, or `Unknown` |
| `Residence` | the first place listed in the `Residence` column |
| `Lives region` | the cleaned province, state, or country |
| `Lives in Canada` | `Canada`, `Elsewhere`, or `Unknown` |
| `Nominations` | how many times the author was nominated |

We use the words `Unknown` instead of `"-"`, and `Canada`/`Elsewhere` instead of `True`/`False`, so that the file is easy to read for anyone who opens it.

First, we need the number of nominations for each author. This is the same counting loop as in Week 6.

In [ ]:
nomination_counts = {}

for book in nominees:
    author = book["Author"]
    if author in nomination_counts:
        nomination_counts[author] = nomination_counts[author] + 1
    else:
        nomination_counts[author] = 1

print(nomination_counts["Michael Christie"])

Next, a small function that turns a region into `Canada`, `Elsewhere`, or `Unknown`. It uses the same test as our other calculations this week.

In [ ]:
def canada_or_elsewhere(region):
    """ Return "Canada" if region is in canadian_regions, "Unknown" if
        region is "-", and "Elsewhere" otherwise.
    """
    if region == "-":
        return "Unknown"
    elif region in canadian_regions:
        return "Canada"
    else:
        return "Elsewhere"


print(canada_or_elsewhere("ON"))
print(canada_or_elsewhere("Sri Lanka"))
print(canada_or_elsewhere("-"))

Now we write the file, one row per author. For each author we look up their cleaned regions in `born_regions` and `home_regions`, and replace any `"-"` with `Unknown`.

In [ ]:
with open("giller_authors.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Author", "Born", "Born region", "Born in Canada",
                     "Residence", "Lives region", "Lives in Canada", "Nominations"])

    for author in born_regions:
        # check this author has a residence too, so we don't get a KeyError
        if author in home_regions:
            born = birthplaces[author]
            parts = residences[author].split(" and ")
            residence = parts[0]

            born_region = born_regions[author]
            lives_region = home_regions[author]

            born_in_canada = canada_or_elsewhere(born_region)
            lives_in_canada = canada_or_elsewhere(lives_region)

            # replace "-" with "Unknown" so the file is easier to read
            if born == "-":
                born = "Unknown"
                born_region = "Unknown"
            if residence == "-":
                residence = "Unknown"
                lives_region = "Unknown"

            writer.writerow([author, born, born_region, born_in_canada,
                             residence, lives_region, lives_in_canada,
                             nomination_counts[author]])

print("Saved giller_authors.csv")

Open `giller_authors.csv` in JupyterLab's file browser to check the result. We'll use this file to start learning **pandas**, a library for working with tables.

## 7. Try it yourself

1. Spreadsheets typed by people contain mistakes: misspelled cities, or a city recorded in the wrong province. Print every birthplace and residence whose region is `ON` and check them against what you know (or can look up). Did you find any errors? Should we fix them in our code or in the spreadsheet?
2. Change the code that builds `home_regions` so that it uses the **last** place listed instead of the first. How do the four groups in section 4 change? Which rule do you think is fairer?
3. Use `born_regions` and `home_regions` to compare birthplaces and residences for Atlantic Canada (`NB`, `NL`, `NS`, `PE`): count how many nominees were born in an Atlantic province, and how many live in one. Do more nominees leave the Atlantic provinces than move there?
4. Repeat Steps 2 to 4 of section 5 using `home_regions` instead of `born_regions`, to compare where the nominees **live** with the population of each province. Which province gains the most nominees compared with where they were born? Which loses the most?